# Tool Output Harmonization

Harmonize raw outputs of mass spectrometry data processing tools.

In [1]:
# Imports
import os
from tqdm import tqdm
import pandas as pd
from benchmarking.harmonizer.harmonization import harmonize

pd.set_option("mode.chained_assignment", None)
tqdm.pandas()

##### Specify directories for harmonization, the tool type for each directory, the full output path, and output type

A directory should contain the following types of raw files for a given tool type
- **mzmine** - Quant table (.csv) from GBPS-FBMN export module used for IIMN, and corresponding MS2 peaklist file (.mgf)
- **metaboscape** - Collapsed feature table output (.csv) and corresponding MS2 peaklist file (.mgf)
- **msdial**:
    - *single* - Single feature table (.csv) with collapsed row features and peak values as well as chromatogram peak areas/heights for a single sample
    - *multi* - Peak master (.csv) file with spectra and collapsed row features and peak values (.csv) file with chromatogram peak areas/heights for multiple samples
    - *multi-combined* - Singular collapsed feature table output (.csv) with chromatogram peak areas/heights, spectra, and collapsed row features (e.g. m/z, rt, etc.)

##### Fetch default directories to be harmonized if directories not specified

In [2]:
input_tool_types_mapping_public = {
    "MSV000090327": "msdial-multi",
    "MSV000091642": "msdial-multi",
    "MSV000095813": "msdial-multi",
    "MSV000097967": "msdial-multi",
    "MSV000097015": "msdial-multi",
    "MSV000096291": "msdial-multi",
    "MSV000096189": "msdial-multi-combined",
    "ST002402": "msdial-multi",
    "MSV000084402": "msdial-single",
    "MTBLS12332": "msdial-multi-combined",
}

input_tool_types_mapping_internal = {
    "MSV000098263": "msdial-multi-combined",
    "NIST_SRM": "msdial-multi",
    "plant_spikein": "msdial-multi-combined",
}

# Processing the public datasets

In [3]:
# Public datasets
base_dir_public = "../data/public_dataset/"
dataset_paths_public = [
    os.path.realpath(os.path.join(base_dir_public, entry))
    for entry in os.listdir(base_dir_public)
]

In [4]:
# Create path list for input directories
dataset_paths_public_filtered = [
    os.path.join(path, "raw")
    for path in dataset_paths_public
    if any(dataset in path for dataset in input_tool_types_mapping_public.keys())
]

input_directories_public = []

for base_dir in dataset_paths_public_filtered:
    for entry in os.listdir(base_dir):
        path = os.path.realpath(os.path.join(base_dir, entry))

        # Count files in the directory
        if os.path.isdir(path) and len(os.listdir(path)) > 0:
            input_directories_public.append(path)

In [5]:
# Create path list for output directories
output_paths_public = []

for path in input_directories_public:
    split_path = path.split("/")
    dataset_id = split_path[-3]
    tool_type = split_path[-1]

    file_name = f"{dataset_id}_{tool_type}_harmonized.parquet"

    prefix = "/".join(split_path[:-2])

    os.makedirs(os.path.join(prefix, "harmonized"), exist_ok=True)

    output_path = os.path.join(prefix, "harmonized", file_name)
    output_paths_public.append(output_path)

In [6]:
input_tool_types_public = []

for path in input_directories_public:
    split_path = path.split("/")
    dataset_id = split_path[-3]
    tool_type = split_path[-1]

    if tool_type == "msdial":
        input_tool_types_public.append(input_tool_types_mapping_public[dataset_id])
    else:
        input_tool_types_public.append(tool_type)

In [7]:
# Harmonize datasets
for input_directory, input_tool_type, output_path in tqdm(
    zip(
        input_directories_public,
        input_tool_types_public,
        output_paths_public,
    ),
    total=len(input_directories_public),
):
    if not os.path.exists(output_path):
        harmonize(input_directory, input_tool_type, output_path)

  7%|▋         | 2/30 [00:00<00:02, 12.89it/s]

100%|██████████| 30/30 [00:47<00:00,  1.59s/it]


# Groundtruth datasets

In [8]:
# Internal datasets
base_dir_internal = "../data/groundtruth_dataset/"
dataset_paths_internal = [
    os.path.realpath(os.path.join(base_dir_internal, entry))
    for entry in os.listdir(base_dir_internal)
]

In [9]:
# Create path list for input directories
dataset_paths_internal_filtered = [
    os.path.join(path, "raw")
    for path in dataset_paths_internal
    if any(dataset in path for dataset in input_tool_types_mapping_internal.keys())
]

input_directories_internal = []

for base_dir in dataset_paths_internal_filtered:
    for entry in os.listdir(base_dir):
        path = os.path.realpath(os.path.join(base_dir, entry))

        # Count files in the directory
        if os.path.isdir(path) and len(os.listdir(path)) > 0:
            input_directories_internal.append(path)

In [10]:
# Create path list for output directories
output_paths_internal = []

for path in input_directories_internal:
    split_path = path.split("/")
    dataset_id = split_path[-3]
    tool_type = split_path[-1]

    file_name = f"{dataset_id}_{tool_type}_harmonized.parquet"

    prefix = "/".join(split_path[:-2])

    os.makedirs(os.path.join(prefix, "harmonized"), exist_ok=True)

    output_path = os.path.join(prefix, "harmonized", file_name)
    output_paths_internal.append(output_path)

In [11]:
input_tool_types_internal = []

for path in input_directories_internal:
    split_path = path.split("/")
    dataset_id = split_path[-3]
    tool_type = split_path[-1]

    if tool_type == "msdial":
        input_tool_types_internal.append(input_tool_types_mapping_internal[dataset_id])
    else:
        input_tool_types_internal.append(tool_type)

In [12]:
# Harmonize datasets
for input_directory, input_tool_type, output_path in tqdm(
    zip(
        input_directories_internal,
        input_tool_types_internal,
        output_paths_internal,
    ),
    total=len(input_directories_internal),
):
    if not os.path.exists(output_path):
        harmonize(input_directory, input_tool_type, output_path)

  0%|          | 0/9 [00:00<?, ?it/s]

100%|██████████| 9/9 [00:51<00:00,  5.73s/it]
